## LORA training/testing pipeline — Task 3 (Provision-Type Classification / LEDGAR)

Given one contract **provision**, classify it into **one of 100 label types** (e.g. *Governing Laws*, *Notices*, *Terminations*). This mirrors the Task 1 / Task 2 notebooks; where T1 answered Yes/No and T2 emitted JSON, T3 emits a single label string.

**Scope of this notebook (for now): only Step 3 of [docs/task_3/TASK3_PLAN.md](docs/task_3/TASK3_PLAN.md)** — turn the downloaded LEDGAR CSVs into the training/validation JSONL files, in the same schema as T1/T2. Training/eval cells (Steps 4–6) will be added later.

What this notebook does:
1. Load the LexGLUE LEDGAR splits produced by `scripts/download_ledgar.py`.
2. Build the shared instruction (with the full 100-label menu) + prompt template.
3. **Stratified-subsample** train and validation so every label gets a fair share (LEDGAR is 137× imbalanced — see [docs/task_3/TASK3_EDA_FINDINGS.md](docs/task_3/TASK3_EDA_FINDINGS.md)).
4. Emit `ledgar_task3_train.jsonl` + `ledgar_task3_validation.jsonl` and sanity-check them.

It is **environment-aware** (runs unchanged locally or on Kaggle): reads go through `DATA_DIR`, writes through `WORK_DIR`.

In [ ]:
# --- Pin to a single GPU BEFORE torch is imported anywhere ---
# Kaggle "GPU T4 x2" exposes 2 GPUs. device_map="auto" then shards the model
# across cuda:0/cuda:1. At loss time TRL's _chunked_cross_entropy_loss builds the
# label mask on cuda:0 while the final hidden_states/lm_head live on cuda:1 ->
# "indices should be either on cpu or on the same device as the indexed tensor
# (cuda:1)". An 8B model in 4-bit (~5-6 GB) fits in ONE T4 (16 GB), so hide GPU 1.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# --- Kaggle: install the library versions this notebook expects (no-op locally) ---
# The Kaggle base image ships older trl/peft; pin trl 1.x so SFTConfig,
# completion_only_loss and processing_class are available.
if os.path.exists("/kaggle"):
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.55.4",   # MUST be <4.56: the 4.56 "core_model_loading"
                                              # threaded loader breaks bnb 4-bit -> full fp16 load -> OOM
                    "bitsandbytes==0.46.1",
                    "accelerate==1.7.0",
                    "peft==0.15.2",
                    "trl==0.20.0",
                    "datasets"], check=True)


In [ ]:
import sys; print("UTF-8 mode:", sys.flags.utf8_mode)

# Step 3.0 : Environment config + imports

Same pattern as the T1/T2 notebooks. Locally, LEDGAR lives at `data/LEDGAR/`; on Kaggle it mounts read-only and generated JSONL must be written to the writable `WORK_DIR` (`/kaggle/working`).

In [ ]:
# --- Environment config: run unchanged locally OR on Kaggle ---
import os
from pathlib import Path

ON_KAGGLE = Path("/kaggle").exists()

if ON_KAGGLE:
    # Read-only mounted dataset. NOTE: set this slug when staging LEDGAR for Kaggle
    # (Step 4) — it must match the LEDGAR dataset-metadata id / mounted folder.
    DATA_DIR = Path("/kaggle/input/ledgar-lexglue")
    # Only this dir is writable AND persisted as kernel output:
    WORK_DIR = Path("/kaggle/working")
else:
    # LEDGAR sits in its own folder, sibling to CUAD_v1.
    DATA_DIR = Path(os.getenv("LEDGAR_DIR", str(Path(os.getenv("DATA_DIR", "data")) / "LEDGAR")))
    WORK_DIR = Path(".")

print(f"ON_KAGGLE={ON_KAGGLE} | DATA_DIR={DATA_DIR} | WORK_DIR={WORK_DIR}")

In [ ]:
import json
import pandas as pd
from collections import Counter

# Reproducibility + sampling knobs (see TASK3_EDA_FINDINGS.md, Finding 1).
SEED = 42
TRAIN_PER_LABEL = 100   # ~100 x 100 labels -> ~10k train examples (rare labels cap lower)
VAL_PER_LABEL = 20      # ~20 x 100 labels  -> ~2k validation examples

pd.set_option("display.max_colwidth", 100)

# Step 3.1 : Load the LEDGAR splits + label map

`scripts/download_ledgar.py` wrote three flat CSVs (`text, label, label_name`) and a `labels.json` id→name map. We only need **train** and **validation** here (test is held out for the final exam).

In [ ]:
# The 100 allowed labels, in id order (canonical menu shown to the model).
with open(DATA_DIR / "labels.json", encoding="utf-8") as fh:
    id2label = {int(k): v for k, v in json.load(fh).items()}
LABELS = [id2label[i] for i in range(len(id2label))]
ALLOWED = set(LABELS)
print(f"{len(LABELS)} labels loaded. e.g. {LABELS[:3]} ... {LABELS[-3:]}")

train_full = pd.read_csv(DATA_DIR / "ledgar_train.csv")
val_full = pd.read_csv(DATA_DIR / "ledgar_validation.csv")
print(f"Full train: {len(train_full):>6} rows | Full validation: {len(val_full):>6} rows")
train_full.head(3)

# Step 3.2 : Build the instruction (100-label menu) + prompt template

The model can only pick a label it has been shown, so the instruction lists **all 100 labels**. This is the same string validated in the EDA notebook (~331 tokens) and is reused verbatim for training, validation, and the base-model baseline so every comparison is fair. The `### Instruction / ### Input / ### Response` template matches T1/T2.

In [ ]:
LABEL_LIST_STR = ", ".join(LABELS)
INSTRUCTION = (
    "Classify the following contract provision. "
    f"Answer with exactly one label from this list: [{LABEL_LIST_STR}]."
)
# Same template as T1/T2 (the completion during training = the label string).
PROMPT_TEMPLATE = "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n"

print(f"Instruction length: {len(INSTRUCTION)} chars")
print(INSTRUCTION[:300] + " ...")

# Step 3.3 : Stratified subsample

Training on all 60k rows would drown the model in common labels (*Governing Laws* alone has 3,167). Instead we take ~`TRAIN_PER_LABEL` examples **per label** so every provision type is represented. Labels with fewer examples than the target (e.g. *Books* = 23, *Assigns* = 31) simply take all they have — that per-label floor is expected and macro-F1 will reflect it. We sample **within** LexGLUE's official splits, so there is no leakage to worry about.

In [ ]:
def stratified_sample(df, per_label, seed=SEED):
    """Take up to `per_label` rows for each label, then shuffle the result."""
    parts = []
    for name, group in df.groupby("label_name"):
        n = min(len(group), per_label)
        parts.append(group.sample(n=n, random_state=seed))
    out = pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    return out

train_sample = stratified_sample(train_full, TRAIN_PER_LABEL)
val_sample = stratified_sample(val_full, VAL_PER_LABEL)

print(f"train: {len(train_full)} -> {len(train_sample)} rows "
      f"({train_sample['label_name'].nunique()}/100 labels)")
print(f"val  : {len(val_full)} -> {len(val_sample)} rows "
      f"({val_sample['label_name'].nunique()}/100 labels)")

# Labels that hit the per-label floor (fewer examples than the target).
train_counts = train_sample["label_name"].value_counts()
capped = sorted(train_counts[train_counts < TRAIN_PER_LABEL].index)
print(f"\nLabels below the {TRAIN_PER_LABEL}/label target in train ({len(capped)}): "
      f"{[(c, int(train_counts[c])) for c in capped]}")

# Step 3.4 : Build the JSONL examples

Same schema as T1/T2 — `instruction`, `category`, `input`, `output` — where for T3 both `category` and `output` are the gold label name (kept as a field for per-label eval later).

In [ ]:
def build_examples(df):
    examples = []
    for row in df.itertuples(index=False):
        label = row.label_name
        examples.append({
            "instruction": INSTRUCTION,
            "category": label,        # gold label name (per-label eval hook)
            "input": str(row.text),
            "output": label,          # completion the model must generate
        })
    return examples

train_data = build_examples(train_sample)
val_data = build_examples(val_sample)
print(f"Built {len(train_data)} train and {len(val_data)} validation examples.")

# Show one full example end-to-end (prompt + expected completion).
ex = train_data[0]
print("\n--- Example ---")
print(PROMPT_TEMPLATE.format(instruction="<100-label instruction>", input=ex["input"]) + ex["output"])
print("\ncategory/output:", ex["category"])

# Step 3.5 : Sanity checks

Mirror T2's pre-save checks, adapted for classification:
1. **Every `output` is a valid label** — verbatim one of the 100 allowed labels (this is T3's equivalent of T2's "output parses as JSON").
2. **Label coverage** — every label appears in **train** (required). Validation coverage is *reported*, not required: the EDA showed `Books` is absent from the validation split (see TASK3_EDA_FINDINGS.md).
3. **Per-label counts** look sane (no empty inputs).

In [ ]:
def check(data, name, require_full_coverage):
    print(f"=== {name} ({len(data)} examples) ===")
    outputs = [e["output"] for e in data]

    # 1. every output is a valid, allowed label
    bad = [o for o in outputs if o not in ALLOWED]
    assert not bad, f"{len(bad)} outputs are not in the 100-label list! e.g. {bad[:5]}"
    print(f"  [OK] all {len(outputs)} outputs are valid labels")

    # 2. no empty / blank inputs
    empty = [e for e in data if not str(e["input"]).strip()]
    assert not empty, f"{len(empty)} examples have empty input text"
    print(f"  [OK] no empty inputs")

    # 3. label coverage
    present = set(outputs)
    missing = sorted(ALLOWED - present)
    print(f"  coverage: {len(present)}/100 labels present")
    if missing:
        print(f"  {'[FAIL]' if require_full_coverage else '[warn]'} missing labels: {missing}")
    if require_full_coverage:
        assert not missing, f"{name} is missing {len(missing)} labels: {missing}"

    # 4. per-label count summary
    counts = Counter(outputs)
    print(f"  per-label counts: min={min(counts.values())} "
          f"max={max(counts.values())} mean={sum(counts.values())/len(counts):.1f}")
    print()

check(train_data, "train", require_full_coverage=True)
check(val_data, "validation", require_full_coverage=False)
print("Sanity checks passed.")

# Step 3.6 : Save to JSONL

Write task-specific filenames (as T2 did) under `WORK_DIR`. On Kaggle `DATA_DIR` is read-only, so generated JSONL goes to the writable `WORK_DIR/ledgar/...`; locally that resolves to the repo root.

In [ ]:
def save_jsonl(data, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for entry in data:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

LEDGAR_TRAIN_PATH = WORK_DIR / "ledgar" / "train"
LEDGAR_VALIDATION_PATH = WORK_DIR / "ledgar" / "validation"
LEDGAR_TRAIN_PATH.mkdir(parents=True, exist_ok=True)
LEDGAR_VALIDATION_PATH.mkdir(parents=True, exist_ok=True)

train_file = LEDGAR_TRAIN_PATH / "ledgar_task3_train.jsonl"
val_file = LEDGAR_VALIDATION_PATH / "ledgar_task3_validation.jsonl"
save_jsonl(train_data, train_file)
save_jsonl(val_data, val_file)
print(f"Saved {len(train_data)} train -> {train_file}")
print(f"Saved {len(val_data)} validation -> {val_file}")

# Step 4 : QLoRA fine-tuning with completion-only loss

Same mechanism as Task 1 / Task 2, same base model, the Task 3 deltas:

- **Separate adapter output:** `new_model_name = "llama-3.1-8B-ledgar-task3"`.
- **Completion is a single label string** (e.g. `Governing Laws`), not `Yes`/`No` (T1) or a JSON object (T2).
- **`MAX_SEQ_LENGTH = 1024`, enforced by trimming the provision text** rather than by truncating the assembled sequence — `build_prompt()` is the single place prompts are made, for training *and* evaluation. The 100-label instruction is 331 tokens (+ ~9 scaffolding) of every prompt, so the label at the tail is exactly what a naive sequence truncation eats.

**Completion-only loss** works exactly as in T1/T2: the dataset is mapped to `prompt` (everything up to and including `### Response:\n`) + `completion` (the label string), and `SFTConfig(completion_only_loss=True)` masks every prompt token so the loss signal concentrates entirely on producing the right label.

> **trl 1.x API note:** `SFTTrainer` takes an `SFTConfig` (not `TrainingArguments`), the tokenizer is passed as `processing_class=`, and `max_seq_length` moved into the config as `max_length`.

## Run history — three failures, three different causes

| Run | Symptom | Root cause | Fix |
| :--- | :--- | :--- | :--- |
| **v10** | NaN loss immediately after `Starting training...` | `max_length=1024` truncated the **assembled sequence**. The label sits at the tail, so the 70 longest rows lost their completion → a `batch=1` micro-batch with every token masked (`-100`) | raise to 2048 (v11), then properly: trim the **input text** instead, so the label always survives |
| **v11** | Killed at 43,200 s, exit 137, 750 MB of checkpoints and nothing else | **`bf16` on a T4.** Turing (cc 7.5) has no bf16 tensor cores — `torch.cuda.is_bf16_supported()` returns `True` anyway, so every matmul fell back to fp32 kernels (~8 vs ~65 TFLOPS). 12.8 s/example ⇒ ~35 h/epoch | `fp16=True` + `bnb_4bit_compute_dtype=torch.float16`, asserted by pre-flight check **1b** |
| **v12** | `OutOfMemoryError: tried to allocate 1.81 GiB` in `cross_entropy`, 4 min in | Peak VRAM is dominated by the **logits tensor**: `tokens_in_batch × 128,256 vocab`, materialized in fp16 and *again in fp32* by the loss. `group_by_length` puts the longest example in batch 0 by design, so batch 0 was 2 × 1,898 tokens → 1.81 GiB on top of ~13.5 GiB in use on a 14.6 GiB card | cap sequences at 1024 (worst case 2 × 1024 = 2,048 tokens) **and** measure it: pre-flight check **9** runs a real forward+backward on that worst-case batch and reports peak GB |
| **v13** | Stopped by pre-flight check 9 itself: worst-case batch peaked at **14.0 / 15.6 GB** | peft's `prepare_model_for_kbit_training` (run inside `SFTTrainer` for 4-bit models) upcasts **every** fp16 parameter to fp32 — including `embed_tokens` and `lm_head`, 525M params each. That is 2.1 GB apiece for frozen weights that came from an fp16 checkpoint, plus ~1.05 GB for the fp16 copy autocast has to cache to run the lm_head matmul | cell **8b** recasts those two modules back to fp16 (~3.2 GB freed; LayerNorms stay fp32), guarded by pre-flight check **2b** — no frozen fp32 tensor over 100M params |

Each fix was confirmed by the next run's log rather than assumed: v12 showed `Compute dtype OK — fp16=True, bf16=False`, and v13 showed the 1024 cap holding (`max=1017, >1024: 0/9801`). `group_by_length` turns what would be a random mid-run OOM into a step-1 failure, and check 9 now turns it into a pre-flight failure — 4 minutes of GPU time instead of 12 hours.

### What the 1024 cap costs

Trimming the provision text (not the prompt) means only the **input** shrinks; instruction, template and label are always intact. Measured in v13: the instruction + template cost **341 tokens**, leaving `MAX_INPUT_TOKENS = 670` for the provision, and that trimmed **79/9,801 train rows (0.81%)** and **9/1,945 validation rows (0.46%)**. Longest assembled sequence dropped to 1,017 tokens, mean 490. LexGLUE's own BERT baseline truncates LEDGAR at **512** tokens, so this is the conservative end of standard practice.

### Other guards

- **`TimeBudgetCallback`** — prints measured pace (and projected epoch time) at steps 5/25/100, and stops training at `TRAIN_TIME_BUDGET_S = 7 h` so the adapter save and eval always run inside Kaggle's 12 h kill. Early stops are recorded as `stopped_on_time_budget` in `train_metrics.json` / `eval_metrics.json`.
- **batch 2 × grad-accum 4** — effective batch stays 8; batch 1 wasted the GPU on bitsandbytes' per-weight NF4 dequantization, which is a fixed cost per forward pass regardless of batch size.
- `save_steps` 100 → 250 with `save_total_limit=1`: v11's 750 MB output was almost entirely redundant checkpoints.

**Pace and memory remain projections until the next run's log confirms them** — check 9's measured peak and the first `[pace]` line are the two numbers to read.

- Note : before execution of the cell below run to the terminal `$env:HF_TOKEN=your_hf_token`

In [ ]:
# Diagnostic: confirm WHICH account the token belongs to and whether it can access the gated repo.
# A 403 "not in the authorized list" means the token is valid but this account lacks access.
import os
from huggingface_hub import whoami, auth_check
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

def get_hf_token():
    if ON_KAGGLE:
        # 1) Headless path: token staged as a file in a PRIVATE mounted dataset.
        #    This is the CLI-only workflow (see kaggle/README.md "HF token"): the token
        #    rides along as a dataset_source in kernel-metadata.json, so a `kernels push`
        #    carries it. No web-UI secret to attach -> nothing for a web "Save Version"
        #    to clobber (which is what silently emptied dataset_sources before).
        for p in Path("/kaggle/input").glob("*/hf_token.txt"):
            tok = p.read_text(encoding="utf-8").strip()
            if tok:
                return tok
        # 2) Fallback: Kaggle Secrets vault (needs the one-time Add-ons -> Secrets attach).
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            return None
    from dotenv import load_dotenv
    load_dotenv()
    return os.getenv("HF_TOKEN")

hf_token = get_hf_token()
assert hf_token, "HF_TOKEN not found (Kaggle private-dataset file, Kaggle Secret, or local .env)"

# Base model for the real (non-smoke-test) run. 8B in 4-bit NF4 is ~5-6 GB of weights,
# which fits fully in a single T4 (16 GB) VRAM with no CPU/disk offload. Smoke-test was "meta-llama/Llama-3.2-1B".
MODEL_ID = "meta-llama/Meta-Llama-3.1-8B"

# 1) Which account is this token? Request access on the model page with THIS exact account.
me = whoami(token=hf_token)
print(f"Token belongs to: {me['name']}  (type: {me.get('type')})")

# 2) Does that account actually have access to the gated repo?
try:
    auth_check(MODEL_ID, token=hf_token)
    print(f"✅ Access granted to {MODEL_ID} — you can run the load cell below.")
except GatedRepoError:
    print(f"❌ Still gated for account '{me['name']}'.")
    print(f"   -> Visit https://huggingface.co/{MODEL_ID} while logged in as '{me['name']}', "
          f"accept the license, and wait for approval.")
    print(f"   -> Or use the ungated mirror: model_name = 'unsloth/Meta-Llama-3.1-8B'")
except HfHubHTTPError as e:
    print(f"❌ Auth/HTTP error (likely an invalid or expired token): {e}")


In [ ]:
import time
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig   # trl 1.x: SFTConfig replaces TrainingArguments here;
                                        # DataCollatorForCompletionOnlyLM was removed (see 5b/7).
import os

def get_hf_token():
    if ON_KAGGLE:
        # Headless path: token staged as a file in a PRIVATE mounted dataset (see the
        # diagnostic cell above and kaggle/README.md). Falls back to the Secrets vault.
        for p in Path("/kaggle/input").glob("*/hf_token.txt"):
            tok = p.read_text(encoding="utf-8").strip()
            if tok:
                return tok
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            return None
    from dotenv import load_dotenv
    load_dotenv()  # reads .env from the current working dir (project root)
    return os.getenv("HF_TOKEN")

hf_token = get_hf_token()
assert hf_token, "HF_TOKEN not found (Kaggle private-dataset file, Kaggle Secret, or local .env)"

from huggingface_hub import login
login(token=hf_token)

# 1. Configuration
# Base model for the real (non-smoke-test) run. 8B in 4-bit NF4 is ~5-6 GB of weights,
# which fits fully in a single T4 (16 GB) VRAM with no CPU/disk offload. Smoke-test was "meta-llama/Llama-3.2-1B".
model_name = "meta-llama/Meta-Llama-3.1-8B"
new_model_name = "llama-3.1-8B-ledgar-task3"   # separate adapter output (T3 / LEDGAR)

# MAX_SEQ_LENGTH = 1024, enforced by trimming the PROVISION TEXT (see build_prompt below)
# rather than by truncating the assembled sequence. That distinction is the whole story of
# runs v10-v12 on this notebook:
#   v10  let trl truncate the assembled sequence at 1024. The label sits at the TAIL, so the
#        70 longest rows lost their completion -> micro-batch with every token masked -> NaN loss.
#   v11  raised the cap to 2048 so nothing was truncated, and hit Kaggle's 12 h wall instead
#        (bf16-on-Turing; see COMPUTE_DTYPE below).
#   v12  fixed the dtype but kept 2048 with batch 2 -> CUDA OOM inside cross_entropy on step 1.
#        Peak VRAM here is dominated by the LOGITS tensor: tokens_in_batch x 128,256 vocab,
#        materialized in fp16 and again in fp32 by the loss. group_by_length deliberately puts
#        the single longest example in batch 0, which was 2 x 1,898 tokens -> a 1.81 GiB fp32
#        allocation on top of ~13.5 GiB already in use on a 14.6 GiB card.
# Trimming the input text keeps the label intact (it is appended *after* the text) and bounds the
# worst-case micro-batch at 2 x 1024 tokens. Only ~0.7% of train rows are affected (EDA Finding 3:
# 99% of prompts are already under 925 tokens), and LexGLUE's own BERT baseline truncates LEDGAR
# at 512 tokens — so this is a conservative cap, not a shortcut. The exact count is printed below.
MAX_SEQ_LENGTH = 1024

# COMPUTE_DTYPE = fp16, NOT bf16. WHY: Kaggle's T4 is Turing (sm_75) and has NO bf16
# tensor cores — bf16 only arrived with Ampere (sm_80). torch.cuda.is_bf16_supported()
# still returns True on a T4 (it reports the CUDA-level emulation path), so bf16 is
# accepted silently and every matmul falls back to fp32 kernels: ~8 TFLOPS instead of
# the ~65 TFLOPS fp16 tensor cores deliver. That is what killed run v11 — 12.8 s per
# example (~102 s per optimizer step), i.e. ~35 h for one epoch inside Kaggle's 12 h
# wall clock, so the kernel was hard-killed at 43200 s (exit 137) with only mid-run
# checkpoints as output. fp16 is the correct compute dtype for this GPU.
COMPUTE_DTYPE = torch.float16

# 2. QLoRA Config (4-bit loading to fit on consumer GPU)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

# 3. Load Base Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,  # non-quantized modules in fp16 too (transformers 4.55 spelling)
    device_map={"": 0},   # whole model on GPU 0 (GPU 1 hidden in the top cell); 8B/4-bit fits in one T4
    token=hf_token
)
model.config.use_cache = False # Silence warnings during training

# 4. Load Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, token=hf_token)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right" # Fix for fp16

# 4b. Prompt construction — ONE definition, shared by training, evaluation and (later) the
# baseline notebook, so every comparison sees byte-identical prompts. The token budget is
# derived from the real tokenizer instead of hardcoded: whatever MAX_SEQ_LENGTH is left after
# the 100-label instruction, the template scaffolding, the longest label and a small margin
# belongs to the provision text.
_overhead_tokens = len(tokenizer(PROMPT_TEMPLATE.format(instruction=INSTRUCTION, input=""))["input_ids"])
_longest_label_tokens = max(len(tokenizer(l, add_special_tokens=False)["input_ids"]) for l in LABELS)
MAX_INPUT_TOKENS = MAX_SEQ_LENGTH - _overhead_tokens - _longest_label_tokens - 8   # 8 = EOS + slack

def truncate_input(text):
    """Trim a provision to MAX_INPUT_TOKENS so the assembled sequence never exceeds
    MAX_SEQ_LENGTH. Trimming the INPUT (not the assembled prompt) is what guarantees the
    completion at the tail survives — the v10 NaN-loss failure was the opposite."""
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    return text if len(ids) <= MAX_INPUT_TOKENS else tokenizer.decode(ids[:MAX_INPUT_TOKENS])

def build_prompt(instruction, input_text):
    """The single prompt construction used everywhere. Ends with '### Response:\\n', which is
    exactly where completion_only_loss stops masking."""
    return PROMPT_TEMPLATE.format(instruction=instruction, input=truncate_input(input_text))

print(f"Prompt budget: {_overhead_tokens} instruction+template tokens + {MAX_INPUT_TOKENS} "
      f"provision tokens + <= {_longest_label_tokens} label tokens <= {MAX_SEQ_LENGTH}")

# 5. Load Dataset (load the same files that were saved in Step 3.6)
dataset = load_dataset("json", data_files={
    "train":      str(LEDGAR_TRAIN_PATH / "ledgar_task3_train.jsonl"),
    "validation": str(LEDGAR_VALIDATION_PATH / "ledgar_task3_validation.jsonl"),
})

# How much data the cap actually touches — a data decision has to be logged, not assumed.
for split in ("train", "validation"):
    trimmed = sum(len(tokenizer(t, add_special_tokens=False)["input_ids"]) > MAX_INPUT_TOKENS
                  for t in dataset[split]["input"])
    print(f"{split}: {trimmed}/{len(dataset[split])} provisions trimmed to {MAX_INPUT_TOKENS} tokens "
          f"({100 * trimmed / len(dataset[split]):.2f}%)")

# 5b. Convert instruction/input/output -> prompt/completion.
# trl 1.x replaces DataCollatorForCompletionOnlyLM with SFTConfig(completion_only_loss=True):
# when the dataset has `prompt` + `completion` columns, SFTTrainer masks the prompt tokens
# automatically so only the label answer contributes to the loss.
def to_prompt_completion(ex):
    return {"prompt": build_prompt(ex["instruction"], ex["input"]),
            "completion": ex["output"]}

dataset = dataset.map(
    to_prompt_completion,
    remove_columns=dataset["train"].column_names,
)

# 6. LoRA Configuration
peft_config = LoraConfig(
    r=16,       # Rank (Higher = more parameters to train, 16-64 is standard)
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

# 7. SFTConfig (trl 1.x: replaces TrainingArguments AND the completion-only collator)
sft_config = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    # batch 1 -> 2 (grad-accum 8 -> 4, so the effective batch stays 8). bitsandbytes pays
    # the NF4 dequantization cost per weight per forward pass no matter how many rows are
    # in the batch, so batch_size=1 spent most of the T4 on dequantizing rather than on
    # matmuls. Kept at 2: worst-case VRAM scales with tokens per micro-batch (2 x 1024 here),
    # and pre-flight check 9 now *measures* that worst case instead of trusting this comment.
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    group_by_length=True,                   # batch similar lengths -> far less padding waste
    gradient_checkpointing=True,            # the big memory saver
    gradient_checkpointing_kwargs={"use_reentrant": False},  # correct grads with PEFT
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,                              # T4 = Turing: fp16 tensor cores, no bf16 (see COMPUTE_DTYPE)
    logging_steps=25,
    save_steps=250,                         # v11 wrote a 173 MB checkpoint every 100 steps for nothing
    save_total_limit=1,
    optim="paged_adamw_32bit",
    max_length=MAX_SEQ_LENGTH,
    completion_only_loss=True,
    packing=False,
    report_to="none",
)


# 7b. Wall-clock guard + pace report.
# Kaggle hard-kills a kernel at 12 h (exit 137); run v11 hit that mid-training, so the
# output held checkpoints but no adapter and no eval — 12 h for zero deliverable. This
# callback (a) prints the measured pace and projected epoch time within the first minutes
# so a too-slow config is obvious immediately, and (b) stops training at TRAIN_TIME_BUDGET_S
# so the adapter save and the validation pass always get to run. Stopping early is recorded
# in train_metrics.json — a partial epoch is a valid, documented experiment; a killed kernel
# is not.
TRAIN_TIME_BUDGET_S = 7 * 3600   # of Kaggle's 12 h; leaves room for save + the eval pass

class TimeBudgetCallback(TrainerCallback):
    """Report training pace early and stop cleanly once the wall-clock budget is spent."""

    def __init__(self, budget_seconds):
        self.budget_seconds = budget_seconds
        self.start = None
        self.stopped_early = False

    def on_train_begin(self, args, state, control, **kwargs):
        self.start = time.time()

    def on_step_end(self, args, state, control, **kwargs):
        elapsed = time.time() - self.start
        if state.global_step in (5, 25, 100):
            per_step = elapsed / state.global_step
            print(f"[pace] step {state.global_step}: {per_step:.1f}s/step -> full epoch "
                  f"({state.max_steps} steps) ~= {per_step * state.max_steps / 3600:.1f} h",
                  flush=True)
        if elapsed > self.budget_seconds:
            print(f"[time budget] {elapsed / 3600:.2f} h elapsed at step {state.global_step}/"
                  f"{state.max_steps} — stopping early so save + eval still run.", flush=True)
            self.stopped_early = True
            control.should_training_stop = True
        return control

time_budget = TimeBudgetCallback(TRAIN_TIME_BUDGET_S)

# 8. Initialize Trainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,   # trl 1.x: was tokenizer=...
    callbacks=[time_budget],
)

# 8b. Undo peft's fp32 upcast of the two BIG frozen modules.
# SFTTrainer runs prepare_model_for_kbit_training() on a 4-bit model, which casts EVERY fp16
# parameter to fp32. For the LayerNorms that is a real stability win costing ~1 MB. For
# embed_tokens and lm_head (128,256 x 4,096 = 525M params each) it costs 2.1 GB apiece and buys
# nothing: both are frozen, and their values came from an fp16 checkpoint, so fp32 stores the
# same numbers in twice the space. Worse, under fp16 autocast the lm_head matmul then has to
# cast its fp32 weight down on every forward, and autocast caches that copy (+1.05 GB).
# Run v13's memory probe measured 14.0 / 15.6 GB peak because of exactly this; casting the two
# back to fp16 frees ~3.2 GB. LayerNorms are deliberately left in fp32.
_recast = []
for _name, _module in trainer.model.named_modules():
    if _name.endswith(("embed_tokens", "lm_head")):
        _bytes_freed = sum(p.numel() * 2 for p in _module.parameters() if p.dtype == torch.float32)
        if _bytes_freed:
            _module.to(COMPUTE_DTYPE)
            _recast.append((_name, _bytes_freed))
if _recast:
    print(f"Recast to {COMPUTE_DTYPE} (frozen, no precision lost): "
          + ", ".join(f"{n} (-{b / 1e9:.2f} GB)" for n, b in _recast))


# Pre-flight checks — verify everything is set up before training

A full QLoRA run is slow and a misconfigured collator fails *silently* (it trains on zero unmasked tokens and the loss never moves). This cell asserts every prerequisite up front so failures surface in seconds, not after an hour:

1. **GPU / VRAM** — CUDA is present and has enough memory for an 8B model in 4-bit.
2. **Model** — actually loaded in 4-bit and `use_cache=False`.
3. **Tokenizer** — `pad_token` set, right-padded.
4. **Datasets** — both splits present, non-empty, with `prompt` / `completion` columns and **every completion is exactly one of the 100 allowed labels** (the Task 3 version of Task 1's Yes/No check and Task 2's JSON-validity check).
5. **Completion-only collator (the critical one)** — the `### Response:\n` template is actually found in tokenized batches and the answer tokens are left **unmasked** (otherwise loss \u2261 0).
6. **Sequence length** — how many examples exceed `MAX_SEQ_LENGTH` and would be truncated (the 100-label instruction is long, so this check is not optional here).
7. **LoRA** — adapters are attached and the base model is frozen (only a tiny % is trainable).

Run this **before** the train cell. If any assertion fails, fix it before training.

In [ ]:
def _ok(msg):   print(f"  [OK]   {msg}")
def _warn(msg): print(f"  [WARN] {msg}")

print("Running pre-flight checks before training...\n")

# 1) Hardware / CUDA — QLoRA needs a GPU; 4-bit 8B wants ~6 GB just for weights.
assert torch.cuda.is_available(), "CUDA not available — QLoRA needs a GPU."
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
cc_major, cc_minor = torch.cuda.get_device_capability(0)
_ok(f"CUDA available — {gpu_name} ({vram_gb:.1f} GB VRAM, compute capability {cc_major}.{cc_minor})")

# 1b) CRITICAL — compute dtype must match the GPU generation. bf16 tensor cores exist only
#     from Ampere (compute capability >= 8.0). On Turing (T4 = 7.5) torch still ACCEPTS bf16
#     — torch.cuda.is_bf16_supported() returns True — but every matmul silently falls back to
#     fp32 kernels (~8 TFLOPS vs ~65 TFLOPS for fp16 tensor cores). That ~8x slowdown is why
#     run v11 needed ~35 h for one epoch and got killed at Kaggle's 12 h wall. Nothing else in
#     the stack complains, so assert it here.
if cc_major < 8:
    assert not sft_config.bf16, (
        f"bf16=True on {gpu_name} (cc {cc_major}.{cc_minor}) — no bf16 tensor cores; matmuls "
        f"drop to fp32 and training runs ~8x slower. Use fp16=True.")
    assert sft_config.fp16, f"Set fp16=True on {gpu_name} — it has fp16 tensor cores."
    assert bnb_config.bnb_4bit_compute_dtype == torch.float16, (
        f"bnb_4bit_compute_dtype={bnb_config.bnb_4bit_compute_dtype} on a pre-Ampere GPU; "
        f"use torch.float16 so the dequantized matmuls hit tensor cores.")
_ok(f"Compute dtype OK — fp16={sft_config.fp16}, bf16={sft_config.bf16}, "
    f"bnb compute dtype={bnb_config.bnb_4bit_compute_dtype}")

# 2) Base model — must actually be 4-bit quantized and have caching off for training.
assert model is not None, "Model not loaded."
is_4bit = getattr(model, "is_loaded_in_4bit", False) or \
          any(p.dtype == torch.uint8 for p in model.parameters())
assert is_4bit, "Model is NOT loaded in 4-bit — check BitsAndBytesConfig / load_in_4bit."
assert model.config.use_cache is False, "model.config.use_cache must be False during training."
_ok(f"Base model '{model_name}' loaded in 4-bit, use_cache=False")

# 2b) No huge FROZEN fp32 tensors. peft's prepare_model_for_kbit_training upcasts every fp16
#     parameter to fp32; on embed_tokens/lm_head that is 2.1 GB each of pure waste (frozen, and
#     the values came from an fp16 checkpoint). Cell 8b recasts them — this proves it stuck.
_big_frozen_fp32 = [(n, p.numel()) for n, p in trainer.model.named_parameters()
                    if p.dtype == torch.float32 and not p.requires_grad and p.numel() > 1e8]
assert not _big_frozen_fp32, (
    f"Frozen fp32 tensors over 100M params: {_big_frozen_fp32} — "
    f"{sum(n for _, n in _big_frozen_fp32) * 2 / 1e9:.1f} GB of VRAM wasted. See cell 8b.")
_ok("No frozen fp32 tensor over 100M params (embed_tokens / lm_head are fp16)")

# 3) Tokenizer — a missing pad_token or left padding silently corrupts batched SFT.
assert tokenizer.pad_token is not None, "Tokenizer has no pad_token."
assert tokenizer.padding_side == "right", f"padding_side must be 'right', got {tokenizer.padding_side!r}."
_ok(f"Tokenizer OK — pad_token={tokenizer.pad_token!r}, padding_side='{tokenizer.padding_side}'")

# 4) Datasets — both splits present, non-empty, prompt/completion schema, and every
#    completion is exactly one of the 100 allowed labels (T3's replacement for the JSON check).
for split in ("train", "validation"):
    assert split in dataset, f"Dataset missing '{split}' split."
    assert len(dataset[split]) > 0, f"'{split}' split is empty."
    missing = {"prompt", "completion"} - set(dataset[split].column_names)
    assert not missing, f"'{split}' split missing columns: {missing}"
    for comp in dataset[split]["completion"]:
        assert comp in ALLOWED, f"Completion is not one of the 100 labels: {comp!r}"
_ok(f"Datasets OK — train={len(dataset['train'])}, val={len(dataset['validation'])}, "
    f"all completions are valid labels")

# 5) CRITICAL — completion-only loss. With trl 1.x + prompt/completion +
#    completion_only_loss=True, SFTTrainer masks the PROMPT tokens (-100) and leaves
#    only the label answer tokens contributing to the loss. Pull one real batch from
#    the trainer's dataloader and confirm BOTH: some tokens unmasked (the answer) AND
#    some tokens masked (the prompt). If nothing is unmasked the loss is ~0 and the
#    model learns nothing; if nothing is masked, the prompt is being trained on too.
#    (This also exercises the group_by_length sampler, so a bad sampler config fails here.)
batch = next(iter(trainer.get_train_dataloader()))
labels_t = batch["labels"]
unmasked = int((labels_t != -100).sum())
masked   = int((labels_t == -100).sum())
assert unmasked > 0, ("All labels masked (-100) — loss would be zero. "
                      "Check completion_only_loss / dataset prompt-completion format.")
assert masked > 0, ("No labels masked — the prompt is not being masked; "
                    "completion_only_loss may be off or the data is not prompt-completion.")
_ok(f"Completion-only loss works — {unmasked} answer token(s) unmasked, "
    f"{masked} prompt token(s) masked in first batch of {labels_t.shape[0]}")

# 6) CRITICAL — sequence length. The completion (label) sits at the very END of the sequence,
#    so any truncation of the ASSEMBLED prompt deletes the LABEL, producing an all-masked
#    micro-batch -> NaN loss (this is exactly what killed run v10). build_prompt() avoids that
#    by trimming the provision TEXT instead, before assembly; this check verifies it worked.
prompts     = dataset["train"]["prompt"]
completions = dataset["train"]["completion"]
lengths = [len(tokenizer(p + c)["input_ids"]) for p, c in zip(prompts, completions)]
over = sum(l > MAX_SEQ_LENGTH for l in lengths)
_ok(f"Token lengths — max={max(lengths)}, mean={sum(lengths)//len(lengths)}, "
    f">{MAX_SEQ_LENGTH}: {over}/{len(lengths)} examples")
assert over == 0, (
    f"{over} train examples exceed MAX_SEQ_LENGTH={MAX_SEQ_LENGTH} (max seen = {max(lengths)}); "
    f"their tail label would be truncated -> all-masked micro-batch -> NaN loss. "
    f"MAX_INPUT_TOKENS={MAX_INPUT_TOKENS} is too generous — lower it or raise MAX_SEQ_LENGTH.")

# 7) LoRA — adapters attached, base model frozen (only a tiny % should train), and the
#    trainable weights in fp32. That last part matters under fp16: the GradScaler refuses
#    to unscale fp16 gradients ("Attempting to unscale FP16 gradients") and the run dies on
#    the first optimizer step. peft casts adapter weights to fp32 for k-bit models by
#    default — this asserts the default actually held.
trainable_params = [p for p in trainer.model.parameters() if p.requires_grad]
trainable = sum(p.numel() for p in trainable_params)
total = sum(p.numel() for p in trainer.model.parameters())
assert trainable > 0, "No trainable parameters — LoRA adapters not attached."
assert trainable < 0.05 * total, f"{100*trainable/total:.2f}% trainable — base model not frozen."
adapter_dtypes = {p.dtype for p in trainable_params}
assert adapter_dtypes == {torch.float32}, (
    f"Trainable params are {adapter_dtypes}, expected fp32 — with fp16=True the GradScaler "
    f"cannot unscale fp16 gradients and training fails on the first optimizer step.")
_ok(f"LoRA attached — trainable {trainable:,} / {total:,} ({100*trainable/total:.3f}%), fp32")

# 8) Budget sanity — how many optimizer steps a full epoch costs, against the wall clock.
#    The exact pace is only knowable once training runs (TimeBudgetCallback prints it at
#    step 5), but the step count makes the target explicit: a step must average under
#    TRAIN_TIME_BUDGET_S / steps_per_epoch seconds for the epoch to finish inside the budget.
effective_batch = sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps
steps_per_epoch = -(-len(dataset["train"]) // effective_batch) * sft_config.num_train_epochs
_ok(f"Budget — {steps_per_epoch:.0f} optimizer steps (effective batch {effective_batch}); "
    f"needs <= {TRAIN_TIME_BUDGET_S / steps_per_epoch:.1f} s/step to finish inside the "
    f"{TRAIN_TIME_BUDGET_S / 3600:.0f} h budget")

# 9) CRITICAL — measured peak VRAM on the WORST-CASE batch. Peak memory here is dominated by
#    the logits tensor (tokens_in_batch x 128,256 vocab), which cross_entropy materializes a
#    second time in fp32 — that 1.81 GiB allocation is what OOM-killed run v12 four minutes in.
#    `batch` above is genuinely the worst case: transformers' LengthGroupedSampler puts the
#    single longest example in batch 0 on purpose, precisely so memory blows up immediately
#    instead of hours later. So run ONE real forward+backward and measure, rather than trusting
#    an estimate. Costs a few seconds and turns a mid-run crash into a pre-flight failure.
probe = {k: batch[k].to(model.device) for k in ("input_ids", "attention_mask", "labels")
         if k in batch}
probe_tokens = probe["input_ids"].numel()
logits_gb = probe_tokens * model.config.vocab_size * 6 / 1e9   # fp16 + fp32 copy
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
# Match the real run: TRL enables gradient checkpointing at trainer init, but make it
# explicit so the probe can never measure a cheaper configuration than training uses.
if sft_config.gradient_checkpointing and not getattr(trainer.model, "is_gradient_checkpointing", False):
    trainer.model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs=sft_config.gradient_checkpointing_kwargs)
trainer.model.train()
try:
    with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
        probe_out = trainer.model(**probe)
    probe_out.loss.backward()
except torch.cuda.OutOfMemoryError as exc:
    raise AssertionError(
        f"OOM on the worst-case batch: {probe_tokens} tokens "
        f"({probe['input_ids'].shape[0]} x {probe['input_ids'].shape[1]}). Logits alone need "
        f"~{logits_gb:.1f} GB (fp16 + the fp32 copy cross_entropy makes). Lower "
        f"per_device_train_batch_size or MAX_SEQ_LENGTH — peak scales with their product."
    ) from exc
peak_gb = torch.cuda.max_memory_allocated() / 1e9
trainer.model.zero_grad(set_to_none=True)
del probe, probe_out
torch.cuda.empty_cache()
_ok(f"Memory probe — worst-case batch ({probe_tokens} tokens, ~{logits_gb:.1f} GB of logits) "
    f"peaked at {peak_gb:.1f} / {vram_gb:.1f} GB")
# Under fp16 the real training path wraps the forward in accelerate's ConvertOutputsToFp32,
# which makes ONE MORE fp32 copy of the logits that this direct-call probe never allocates.
# Add it explicitly rather than discovering it as an OOM at step 1.
accelerate_logits_gb = probe_tokens * model.config.vocab_size * 4 / 1e9
projected_gb = peak_gb + accelerate_logits_gb
assert projected_gb < 0.85 * vram_gb, (
    f"Worst-case batch projects to {projected_gb:.1f} GB (probe {peak_gb:.1f} + "
    f"{accelerate_logits_gb:.1f} GB accelerate fp32 logits copy) of {vram_gb:.1f} GB — too close "
    f"to the limit once optimizer state and fragmentation are added. Halve "
    f"per_device_train_batch_size to 1 and double gradient_accumulation_steps to 8 "
    f"(effective batch stays 8), or lower MAX_SEQ_LENGTH.")
_ok(f"Projected training peak {projected_gb:.1f} / {vram_gb:.1f} GB "
    f"(+{accelerate_logits_gb:.1f} GB for accelerate's fp32 logits copy)")

print("\nAll pre-flight checks passed — safe to run the training cell below.")


In [ ]:
# 9. Train and Save  (run only after the pre-flight checks above pass)
print("Starting training...", flush=True)
train_result = trainer.train()

# Post-training sanity: the loss must actually be a real, finite, non-zero number.
# A training loss that is exactly 0 / NaN means the completion-only masking ate
# every label — exactly the silent failure the pre-flight check guards against.
final_loss = train_result.training_loss
assert final_loss is not None and final_loss == final_loss, f"Training loss is NaN: {final_loss}"
assert final_loss > 0, f"Training loss is {final_loss} — no tokens contributed to the loss."
print(f"Training finished — final training loss: {final_loss:.4f} "
      f"({train_result.global_step} steps, {train_result.metrics.get('epoch', 0):.2f} epochs, "
      f"{train_result.metrics.get('train_runtime', 0) / 3600:.2f} h)", flush=True)
if time_budget.stopped_early:
    print("NOTE: training stopped on the wall-clock budget, not at the end of the epoch — "
          "the adapter below is trained on a PARTIAL epoch (see train_metrics.json).", flush=True)

# Save the adapter to the writable/persisted dir.
# On Kaggle WORK_DIR=/kaggle/working (downloadable output); locally it is the repo root.
save_dir = WORK_DIR / new_model_name
trainer.model.save_pretrained(save_dir)
saved = list(Path(save_dir).glob("adapter_*"))
assert any(p.name == "adapter_model.safetensors" or p.name == "adapter_model.bin" for p in saved), \
    f"No adapter weights found in {save_dir}/ — save may have failed."
assert (Path(save_dir) / "adapter_config.json").exists(), \
    f"adapter_config.json missing in {save_dir}/."
print(f"Model saved to {save_dir}/ — files: {sorted(p.name for p in Path(save_dir).iterdir())}")

# --- Persist training metrics as a downloadable artifact ---
# The final loss otherwise only shows in the Kaggle run log. Write it (plus the key
# hyper-params that produced it) to WORK_DIR so it returns in kaggle_output/ and
# each run is self-describing / reproducible.
train_metrics = {
    "model_name": model_name,
    "new_model_name": new_model_name,
    "final_training_loss": float(final_loss),
    "train_runtime_seconds": train_result.metrics.get("train_runtime"),
    "train_samples_per_second": train_result.metrics.get("train_samples_per_second"),
    "global_step": train_result.global_step,
    "epochs_completed": train_result.metrics.get("epoch"),
    # True = the wall-clock guard cut the epoch short; the adapter saw only part of the
    # training set, so any metric below is a partial-epoch result, not a 1-epoch result.
    "stopped_on_time_budget": time_budget.stopped_early,
    "train_time_budget_seconds": TRAIN_TIME_BUDGET_S,
    "hyperparameters": {
        "num_train_epochs": sft_config.num_train_epochs,
        "per_device_train_batch_size": sft_config.per_device_train_batch_size,
        "gradient_accumulation_steps": sft_config.gradient_accumulation_steps,
        "group_by_length": sft_config.group_by_length,
        "learning_rate": sft_config.learning_rate,
        "weight_decay": sft_config.weight_decay,
        "max_length": sft_config.max_length,
        "optim": sft_config.optim,
        "fp16": sft_config.fp16,
        "bf16": sft_config.bf16,
        "bnb_4bit_compute_dtype": str(bnb_config.bnb_4bit_compute_dtype),
        "completion_only_loss": sft_config.completion_only_loss,
        "packing": sft_config.packing,
        "lora_r": peft_config.r,
        "lora_alpha": peft_config.lora_alpha,
        "lora_dropout": peft_config.lora_dropout,
        "lora_target_modules": list(peft_config.target_modules),
    },
}
train_metrics_path = WORK_DIR / "train_metrics.json"
with open(train_metrics_path, "w", encoding="utf-8") as f:
    json.dump(train_metrics, f, indent=2)
print(f"Wrote training metrics to {train_metrics_path}")


# Step 5 : Evaluate on validation — valid-label rate, accuracy, macro/micro-F1, per-label, confusions

Greedy generation (`do_sample=False`) on the 1,945-row validation set, scored on classification metrics in a deliberate order (see TASK3_NEXT_STEPS.md step 3):

1. **Valid-label rate** — the raw completion, after strip + lowercase (first line only), is exactly one of the 100 labels. T3's analogue of T1's Yes/No and T2's JSON-validity: an unparseable answer counts as a wrong prediction (mapped to `__INVALID__`), never silently forgiven.
2. **Accuracy** — fraction where predicted label == gold label.
3. **Macro-F1** — the headline number; LEDGAR is 137× imbalanced (EDA Finding 1) so the unweighted per-label mean is what matters.
4. **Micro-F1** — for LexGLUE-leaderboard comparison (published BERT-class yardstick ~87–88 micro / ~82 macro).
5. **Per-label precision/recall/F1** and the **top 15 confused label pairs** — watch the known look-alikes (Governing Laws/Jurisdictions, Assigns/Successors, Amendments/Modifications, Waivers/No Waivers).

Metrics are computed with `labels=LABELS` so invalid predictions count against recall without inflating any real label's precision. Results persist to `WORK_DIR` as `eval_metrics.json` / `eval_report.txt` — the same filenames as T1/T2 — so `kaggle kernels output` retrieves them. The base-model (no fine-tune) run of this same evaluation lives in a separate notebook (step 5 of the next-steps doc); the deliverable is the delta.

**Generation is batched** (`EVAL_BATCH_SIZE = 8`), unlike the v11 loop that called `generate()` once per example and paid the full 8B-model launch overhead 1,945 times — hours out of a 12 h budget for ~16 decoded tokens each. Two details make batching correct rather than just fast: the tokenizer switches to **left padding** (with right padding the model would continue from pad tokens instead of from `### Response:\n`), and `max_new_tokens` is derived from the longest label in the menu instead of a hardcoded 16.

In [ ]:
from sklearn.metrics import f1_score, precision_recall_fscore_support

# Evaluate through the trainer's model: that is the PEFT-wrapped model, so the LoRA
# adapter we just trained is unambiguously active (not the bare 4-bit base model).
eval_model = trainer.model
eval_model.config.use_cache = True          # KV cache — pointless during training, essential here
eval_model.gradient_checkpointing_disable() # recompute-instead-of-store is a pure cost at inference
eval_model.eval()

# Generation needs LEFT padding: with right padding the pad tokens sit after the prompt and
# the model continues from padding instead of from "### Response:\n".
tokenizer.padding_side = "left"

# v11 evaluated one example per generate() call: 1,945 sequential calls, each paying the
# full 8B-model launch overhead for ~16 decoded tokens. Batching amortizes that over
# EVAL_BATCH_SIZE rows. Memory is far cheaper here than in training — generate() computes
# logits for one position, not the whole sequence — so the cost is the KV cache
# (~130 KB/token for an 8B model): 8 x 1024 tokens is ~1 GB on top of the weights.
EVAL_BATCH_SIZE = 8
# Longest label in tokens (+2 for the EOS the fine-tuned model emits). Generating past the
# label is wasted decode time; the parser only reads the first line anyway.
MAX_NEW_TOKENS = _longest_label_tokens + 2

# Canonical-label lookup for parsing (metric 1): strip + lowercase, first line only.
LABEL_BY_LOWER = {l.lower(): l for l in LABELS}
INVALID = "__INVALID__"   # sentinel for un-parseable predictions (not in LABELS)

def predict_labels(examples):
    """Greedy-generate for a batch of examples -> [(canonical_label, is_valid), ...]."""
    # build_prompt() — the same function training used, including the same provision-text
    # trimming — so train and eval prompts are byte-identical in construction.
    prompts = [build_prompt(e["instruction"], e["input"]) for e in examples]
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to(eval_model.device)
    with torch.no_grad():
        out = eval_model.generate(
            **enc,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Left padding means every row's continuation starts at the same offset.
    generated = out[:, enc["input_ids"].shape[1]:]
    results = []
    for raw in tokenizer.batch_decode(generated, skip_special_tokens=True):
        text = raw.strip().split("\n", 1)[0].strip()   # first line only
        canonical = LABEL_BY_LOWER.get(text.lower())
        results.append((canonical, True) if canonical is not None else (INVALID, False))
    return results

eval_start = time.time()
gold, pred, valid_flags = [], [], []
for start in range(0, len(val_data), EVAL_BATCH_SIZE):
    chunk = val_data[start:start + EVAL_BATCH_SIZE]
    for ex, (label, is_valid) in zip(chunk, predict_labels(chunk)):
        gold.append(ex["category"])
        pred.append(label)
        valid_flags.append(is_valid)
    if len(gold) % (EVAL_BATCH_SIZE * 25) == 0:
        print(f"  evaluated {len(gold)}/{len(val_data)} "
              f"({time.time() - eval_start:.0f}s elapsed)", flush=True)
eval_runtime = time.time() - eval_start

# --- Aggregate metrics (labels=LABELS: invalid preds count as misses, never FPs) ---
n = len(gold)
valid_label_rate = sum(valid_flags) / n
accuracy = sum(g == p for g, p in zip(gold, pred)) / n
macro_f1 = f1_score(gold, pred, labels=LABELS, average="macro", zero_division=0)
micro_f1 = f1_score(gold, pred, labels=LABELS, average="micro", zero_division=0)
prec, rec, f1, support = precision_recall_fscore_support(
    gold, pred, labels=LABELS, zero_division=0)

# Top confused (gold -> pred) pairs, most frequent first.
conf = Counter((g, p) for g, p in zip(gold, pred) if g != p)
top_conf = conf.most_common(15)

print(f"\nValidation examples : {n}  (evaluated in {eval_runtime / 60:.1f} min)")
print(f"Valid-label rate    : {valid_label_rate:.4f}")
print(f"Accuracy            : {accuracy:.4f}")
print(f"Macro-F1 (headline) : {macro_f1:.4f}")
print(f"Micro-F1            : {micro_f1:.4f}")

print("\nTop 15 confused (gold -> pred) pairs:")
for (g, p), c in top_conf:
    print(f"  {c:3d}  {g}  ->  {p}")

# --- Per-label table (sorted by F1 ascending: worst offenders first) ---
per_label_rows = sorted(
    ({"label": LABELS[i], "precision": prec[i], "recall": rec[i],
      "f1": f1[i], "support": int(support[i])} for i in range(len(LABELS))),
    key=lambda r: r["f1"],
)
hdr = f"{'Label':40s} {'precision':>9s} {'recall':>7s} {'f1':>6s} {'support':>8s}"
tbl = [hdr, "-" * len(hdr)]
for r in per_label_rows:
    tbl.append(f"{r['label']:40s} {r['precision']:9.2f} {r['recall']:7.2f} "
               f"{r['f1']:6.2f} {r['support']:8d}")
per_label_table = "\n".join(tbl)

# --- Persist artifacts (same filenames/convention as T1/T2) ---
eval_metrics = {
    "model_name": new_model_name,
    "task": "task3_provision_classification",
    "n_validation_examples": n,
    "eval_runtime_seconds": eval_runtime,
    # Carried over so a metrics file is never mistaken for a full-epoch result.
    "stopped_on_time_budget": time_budget.stopped_early,
    "max_seq_length": MAX_SEQ_LENGTH,
    "valid_label_rate": float(valid_label_rate),
    "accuracy": float(accuracy),
    "macro_f1": float(macro_f1),
    "micro_f1": float(micro_f1),
    "per_label": {
        LABELS[i]: {"precision": float(prec[i]), "recall": float(rec[i]),
                    "f1": float(f1[i]), "support": int(support[i])}
        for i in range(len(LABELS))
    },
    "top_confusions": [
        {"gold": g, "pred": p, "count": c} for (g, p), c in top_conf
    ],
}
eval_metrics_path = WORK_DIR / "eval_metrics.json"
with open(eval_metrics_path, "w", encoding="utf-8") as f:
    json.dump(eval_metrics, f, indent=2)

eval_report_path = WORK_DIR / "eval_report.txt"
with open(eval_report_path, "w", encoding="utf-8") as f:
    f.write("Task 3 — LEDGAR provision classification (validation)\n\n")
    f.write(f"Validation examples : {n}\n")
    f.write(f"Valid-label rate    : {valid_label_rate:.4f}\n")
    f.write(f"Accuracy            : {accuracy:.4f}\n")
    f.write(f"Macro-F1 (headline) : {macro_f1:.4f}\n")
    f.write(f"Micro-F1            : {micro_f1:.4f}\n")
    if time_budget.stopped_early:
        f.write("\nNOTE: training stopped on the wall-clock budget — partial epoch.\n")
    f.write("\nTop 15 confused (gold -> pred) pairs:\n")
    for (g, p), c in top_conf:
        f.write(f"  {c:3d}  {g}  ->  {p}\n")
    f.write("\nPer-label precision / recall / F1 (worst F1 first):\n")
    f.write(per_label_table + "\n")

print(f"\nWrote eval metrics to {eval_metrics_path}")
print(f"Wrote eval report  to {eval_report_path}")
